# 00 总目录与项目架构

**用途：** 从业务问题出发理解项目一 RAG 和项目二多工具 Agent 的关系，并确认所有核心模块都存在。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


## 1. 这个项目到底解决什么问题

项目一解决“企业资料里有没有答案”：加载企业文档、切分、向量检索、来源引用、低置信拒答和工单草稿。

项目二解决“客服收到问题后应该办什么事”：识别意图、补齐槽位、查询库存、生成报价、估算物流、建立售后草稿、查询项目一 RAG，并在高风险或不确定时暂停给人工。

```text
客户文字/图片
    -> 上下文与客户记忆
    -> LangGraph State
    -> 语义解析（规则优先，LangChain补充）
    -> Pydantic 参数校验
    -> 确定性工具 / RAG / 视觉模型
    -> 审批、图片确认或人工接管
    -> 客户回复 + checkpoint + 日志
```

**成功标准：** 能说清楚“RAG负责知识，工具负责动作，LangGraph负责编排，LangChain负责组件协议，Harness负责模型运行边界”。

In [2]:
core_files = [
    DAY1_ROOT / "rag_components.py",
    DAY1_ROOT / "rag_chat.py",
    PROJECT2_ROOT / "agent_graph.py",
    PROJECT2_ROOT / "langchain_adapter.py",
    PROJECT2_ROOT / "langchain_tools.py",
    PROJECT2_ROOT / "schemas.py",
    PROJECT2_ROOT / "context_manager.py",
    PROJECT2_ROOT / "memory_repository.py",
    PROJECT2_ROOT / "conversation_repository.py",
    PROJECT2_ROOT / "handoff_repository.py",
    PROJECT2_ROOT / "agent_harness.py",
    PROJECT2_ROOT / "model_router.py",
    PROJECT2_ROOT / "vision_service.py",
]
inventory = file_inventory(core_files, DAY1_ROOT)
show_table(inventory)
check("核心源码完整", all(item["存在"] for item in inventory), "13个关键文件均存在")

,文件,存在,大小
0,rag_components.py,True,4546
1,rag_chat.py,True,26517
2,project2\agent_graph.py,True,63702
3,project2\langchain_adapter.py,True,6944
4,project2\langchain_tools.py,True,2163
5,project2\schemas.py,True,5328
6,project2\context_manager.py,True,17395
7,project2\memory_repository.py,True,14851
8,project2\conversation_repository.py,True,15189
9,project2\handoff_repository.py,True,10786


[PASS] 核心源码完整 | 13个关键文件均存在


{'检查项': '核心源码完整', '状态': 'PASS', '说明': '13个关键文件均存在'}

In [3]:
import unittest
suite = unittest.TestLoader().discover(str(PROJECT2_ROOT / "tests"), pattern="test_*.py")
test_count = suite.countTestCases()
check_equal("运行时与集成测试数量", test_count, 67)

tools = [
    "inventory_tool", "quote_tool", "logistics_tool",
    "ticket_tool", "knowledge_tool",
]
show_table([{"工具": item, "作用": role} for item, role in zip(
    tools,
    ["查库存", "报价草稿", "物流估算", "售后草稿", "企业知识检索"],
)])

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] 运行时与集成测试数量 | actual=67, expected=67


,工具,作用
0,inventory_tool,查库存
1,quote_tool,报价草稿
2,logistics_tool,物流估算
3,ticket_tool,售后草稿
4,knowledge_tool,企业知识检索


,工具,作用
0,inventory_tool,查库存
1,quote_tool,报价草稿
2,logistics_tool,物流估算
3,ticket_tool,售后草稿
4,knowledge_tool,企业知识检索


## 2. 已实现与后续规划

| 状态 | 内容 |
| --- | --- |
| 已实现 | RAG、多工具、LangGraph、LangChain、checkpoint、上下文、长期记忆、多会话、HITL、人工客服、Harness、多模态MVP、双人盲审工具 |
| 需要人工完成 | 40张图片两人独立标注和第三人裁决 |
| 尚未实现 | 受控网页搜索、FastAPI服务层、LangSmith正式接入、Skills、sub-agent、完整multi-agent |

### 面试官会问

1. 项目一和项目二有什么本质区别？
2. 为什么不是让大模型直接回答库存和价格？
3. LangChain、LangGraph、RAG和Harness各自处在哪一层？
4. 哪些结论是自动评测证明的，哪些还只是设计？

### 参考答案

1. **项目一和项目二有什么本质区别？** 项目一解决“根据企业资料回答问题”，主链路是检索、生成和来源引用；项目二解决“按销售流程办事”，除了复用项目一RAG，还要维护State、补齐槽位、调用库存/报价/物流/售后工具、暂停审批并在失败时转人工。
2. **为什么不让模型直接回答库存和价格？** 库存和价格属于动态、高风险业务事实。模型只能理解意图并生成结构化计划，最终数据必须由确定性工具读取CSV或业务系统；调用前再用Pydantic和业务规则校验，避免幻觉和越权承诺。
3. **四个组件分别在哪一层？** RAG提供企业知识证据；LangChain统一Document、Retriever、Prompt、结构化输出和Tool协议；LangGraph负责状态、节点、条件路由、checkpoint和interrupt；Harness包住模型调用，控制超时、重试、预算、并发、脱敏和telemetry。
4. **哪些有自动证据？** 30/30两套业务回归、67条运行时/集成测试、checkpoint恢复、人工接管、图片安全门控和16本Notebook均有自动结果。40张公开图片只完成API预跑，尚未完成双人gold，因此不能宣称字段准确率；网页搜索、FastAPI、LangSmith和完整multi-agent仍是后续设计。

**代码落点：** `agent_graph.py`、`langchain_adapter.py`、`tools/knowledge_tool.py`、`agent_harness.py`、`tests/`。

**回答主线：** 先讲业务风险，再讲确定性边界，最后讲评测证据。不要只罗列框架名。